# 06 · Prefill, decode and the roofline
Level L1 · Part 1 · core · ~25 min · builds on 02, 03

## Goal
By the end of this lesson you can predict how TTFT (time to first token: the wait before the answer starts) and TPOT (time per output token: the gap between later tokens) move when the prompt, the batch or the weight precision changes, and you can say which phase is compute-bound and which is memory-bound, using arithmetic intensity.

## Why it matters
- **Without it:** you size a GPU from its headline TFLOPS. That number says how fast the chip does arithmetic. Decode does little arithmetic per byte it reads, so the maths is not its limit. The next cell shows the error.
- **With it:** each phase gets two numbers, the work it does (FLOPs) and the bytes it reads from GPU memory (HBM, the GPU's own high-bandwidth memory). Their ratio, arithmetic intensity, says which limit a phase hits. Then you know which hardware number sets TTFT and which one sets TPOT.
- **When you don't need it:** if a hosted provider serves the model and you only read its published TTFT and TPOT, you do not need this to buy hardware. You still need it to judge whether a latency target is physically possible.

### The wrong estimate: FLOPs only
Take one decode step of the 8B model (16 billion FLOPs, about 2 per weight). Divide by the L4's peak maths and you get a time per token. Then divide the 16 GB of weights by the L4's memory bandwidth, which is the time the step really waits for.

In [1]:
PEAK_FLOPS, BANDWIDTH = 121e12, 300e9          # L4, dense FP16 and memory bandwidth
WEIGHTS = 16e9                                  # 8 billion weights x 2 bytes (FP16)
FLOPS_PER_STEP = 2 * 8e9                        # about 2 FLOPs per weight per token

print(f"FLOPs-only estimate: {FLOPS_PER_STEP / PEAK_FLOPS * 1000:.2f} ms per token "
      f"= {PEAK_FLOPS / FLOPS_PER_STEP:,.0f} tokens/s")
print(f"reading the weights: {WEIGHTS / BANDWIDTH * 1000:.1f} ms per token "
      f"= {BANDWIDTH / WEIGHTS:.1f} tokens/s")

FLOPs-only estimate: 0.13 ms per token = 7,562 tokens/s
reading the weights: 53.3 ms per token = 18.8 tokens/s


**What you should see:** the FLOPs-only estimate says about 7,560 tokens per second for one request. Reading the weights gives about 19 tokens per second. The FLOPs-only estimate is about 400 times too optimistic, so it cannot size anything. The rest of this lesson shows why.

## The idea
Any step needs some arithmetic (FLOPs) and some data movement (bytes read from GPU memory). The step cannot finish faster than either one allows, so its time is at least the larger of the two. **Arithmetic intensity** is FLOPs divided by bytes. Every GPU has a **ridge point**: its peak FLOP/s divided by its bandwidth. A step above the ridge is **compute-bound** (the maths is the limit). A step below it is **memory-bound** (the data reads are the limit). This is the **roofline** model.

## Picture
```mermaid
flowchart LR
  P["Prefill: 1,600 prompt tokens in one pass"] --> PI["Intensity about 1,600 FLOP/byte: above the ridge"]
  PI --> PC["Compute-bound: sets TTFT"]
  D["Decode: one token per step, batch 1"] --> DI["Intensity about 1 FLOP/byte: below the ridge"]
  DI --> DM["Memory-bound: reads all weights each step, sets TPOT"]
```

## Step by step
### Step 1: the model shape and the GPUs
The model is Llama 3.1 8B in FP16, with the shape from lesson 03. The GPU peaks are dense numbers from NVIDIA's product pages (opened 2026-10-21, see RESEARCH.md). **Dense** means the peak without structured sparsity. The "sparse" figures on those pages are about twice these and are not used here.

`ATTN_FLOPS_PER_PAIR = 4 * LAYERS * Q_HEADS * HEAD_DIM` is the attention work per query-key pair: 2 FLOPs for the QK score and 2 for the weights times V, per head per layer.

In [2]:
PARAMS = 8.0e9                                   # about 8 billion weights
LAYERS, Q_HEADS, KV_HEADS, HEAD_DIM = 32, 32, 8, 128
FP16 = 2                                         # bytes per number
KV_PER_TOKEN = 2 * LAYERS * KV_HEADS * HEAD_DIM * FP16   # 131,072 bytes per token (lesson 03)
ATTN_FLOPS_PER_PAIR = 4 * LAYERS * Q_HEADS * HEAD_DIM    # 2 FLOPs for QK, 2 for weights x V, per head per layer

GPUS = {
    "L4":       {"flops": 121e12, "bw": 300e9,   "hbm_gb": 24},
    "H100 SXM": {"flops": 989e12, "bw": 3.35e12, "hbm_gb": 80},
}
for name, g in GPUS.items():
    print(f"{name:9} {g['flops'] / 1e12:>5.0f} TFLOP/s  {g['bw'] / 1e9:>6,.0f} GB/s  "
          f"ridge {g['flops'] / g['bw']:>4.0f} FLOP/byte  {g['hbm_gb']} GB")

L4          121 TFLOP/s     300 GB/s  ridge  403 FLOP/byte  24 GB
H100 SXM    989 TFLOP/s   3,350 GB/s  ridge  295 FLOP/byte  80 GB


### Step 2: FLOPs and bytes for each phase
**Prefill** runs the weights once over all prompt tokens, so each weight read is reused n times. Its attention work is the pair count from lesson 02. **Decode** runs the weights once per step, and that step reads the KV cache of every request in the batch. The batch is how many requests share one weight read.

The Brightside prompt is 1,600 tokens: a 1,500-token policy and a 100-token question (lesson 02). Its 200-token answer brings the cache to 1,800 tokens (lesson 03). The byte counts below include only the weights and the KV cache. Activations and runtime overhead are ignored here, so real memory is tighter than these counts suggest (lesson 05 counts them).

In [3]:
def prefill_cost(n):
    """One pass over n prompt tokens: weights read once; attention pairs n(n+1)/2."""
    flops = 2 * PARAMS * n + ATTN_FLOPS_PER_PAIR * n * (n + 1) // 2
    bytes_read = PARAMS * FP16 + n * KV_PER_TOKEN       # weights, plus the KV cache written
    return flops, bytes_read

def decode_cost(ctx, batch=1, weight_bytes=FP16):
    """One decode step: each request has ctx cached tokens; the weights are read once for the batch."""
    flops = batch * (2 * PARAMS + ATTN_FLOPS_PER_PAIR * ctx)
    bytes_read = PARAMS * weight_bytes + batch * ctx * KV_PER_TOKEN
    return flops, bytes_read

for label, (f, b) in (("prefill, 1,600 tokens", prefill_cost(1600)),
                      ("decode step, 1 request at 1,800 tokens", decode_cost(1800))):
    print(f"{label:40} {f / 1e9:>10,.1f} GFLOP  {b / 1e9:>6.2f} GB  intensity {f / b:>8,.1f} FLOP/byte")

prefill, 1,600 tokens                      26,271.5 GFLOP   16.21 GB  intensity  1,620.7 FLOP/byte
decode step, 1 request at 1,800 tokens         16.9 GFLOP   16.24 GB  intensity      1.0 FLOP/byte


**What you should see:** the prefill does about 1,600 FLOPs per byte it reads, and the decode step about 1. Prefill reuses each weight 1,600 times in one pass. A decode step uses each weight once.

### Step 3: which limit each phase hits
For each phase and GPU, the ideal time is the larger of "FLOPs divided by peak" and "bytes divided by bandwidth". This is a lower bound. A real engine reaches only part of the peak, and lesson 07 measures how much.

In [4]:
def phase_limit(flops, bytes_read, gpu):
    t_math = flops / GPUS[gpu]["flops"]
    t_mem = bytes_read / GPUS[gpu]["bw"]
    bound = "compute-bound" if t_math > t_mem else "memory-bound"
    return bound, max(t_math, t_mem) * 1000        # milliseconds

for gpu in GPUS:
    for label, (f, b) in (("prefill, 1,600 tokens", prefill_cost(1600)),
                          ("decode step, batch 1", decode_cost(1800))):
        bound, ms = phase_limit(f, b, gpu)
        print(f"{gpu:9} {label:22} {bound:14} ideal {ms:8.2f} ms")

L4        prefill, 1,600 tokens  compute-bound  ideal   217.12 ms
L4        decode step, batch 1   memory-bound   ideal    54.12 ms
H100 SXM  prefill, 1,600 tokens  compute-bound  ideal    26.56 ms
H100 SXM  decode step, batch 1   memory-bound   ideal     4.85 ms


**What you should see:** prefill is compute-bound on both GPUs. Decode at batch 1 is memory-bound on both. The decode step's ideal time is about 54 ms on the L4 and about 5 ms on the H100, which matches the weight-read times above.

### Step 4: the trends for TTFT and TPOT
TTFT is set by the prefill. Its work grows with the prompt, and the attention part grows with the square. The next cell prints TTFT on the L4 for five prompt lengths.

In [5]:
print("TTFT (ideal), prompt length on the L4:")
for n in (500, 1_600, 4_000, 8_000, 16_000):
    f, b = prefill_cost(n)
    attn_share = (f - 2 * PARAMS * n) / f
    print(f"  {n:>6,} tokens  {phase_limit(f, b, 'L4')[1]:>8,.0f} ms   attention share {attn_share:4.0%}")

TTFT (ideal), prompt length on the L4:
     500 tokens        67 ms   attention share   1%
   1,600 tokens       217 ms   attention share   3%
   4,000 tokens       564 ms   attention share   6%
   8,000 tokens     1,197 ms   attention share  12%
  16,000 tokens     2,670 ms   attention share  21%


TPOT is set by the decode step. At batch 1 it grows only with the KV cache it reads. The next cell prints TPOT on the L4 for four context lengths.

In [6]:
print("TPOT (ideal), context length at batch 1 on the L4:")
for ctx in (500, 1_800, 8_000, 32_000):
    f, b = decode_cost(ctx)
    print(f"  {ctx:>6,} cached tokens  {phase_limit(f, b, 'L4')[1]:>6.1f} ms per token")

TPOT (ideal), context length at batch 1 on the L4:
     500 cached tokens    53.6 ms per token
   1,800 cached tokens    54.1 ms per token
   8,000 cached tokens    56.8 ms per token
  32,000 cached tokens    67.3 ms per token


**What you should see:** doubling the prompt from 4,000 to 8,000 tokens more than doubles TTFT (564 ms to 1,197 ms), because attention work grows with the square. The 16,000-token prompt takes about 4.7 times as long as the 4,000-token one (2,670 ms). TPOT rises only a little, from 53.6 ms at 500 cached tokens to 67.3 ms at 32,000, and the 1,800-token row sits in between. The 16 GB of weights are read on every step either way.

### Step 5: batching, and why the KV reads can cap decode
Batching adds FLOPs but almost no weight bytes, because the weights are read once per step. Each request still reads its own KV cache, so the intensity climbs towards a limit set by the context.

**Deriving the batch that reaches the ridge.** Let F be the FLOPs per token for one request, W the weight bytes (16 GB), and K the KV bytes per request (context × 131,072). A batch of B gives intensity(B) = B·F / (W + B·K). Set it equal to the ridge R and solve: B = R·W / (F − R·K). The cell's `spare` is F − R·K. If spare ≤ 0 there is no solution (the `None` case): the KV bytes grow faster than the FLOPs.

"Room" is the largest batch the GPU memory holds after the weights.

In [7]:
import math

def batch_for_ridge(ctx, gpu):
    """Batch B where B x FLOPs-per-request / (weights + B x KV bytes) reaches the ridge; None if never."""
    ridge = GPUS[gpu]["flops"] / GPUS[gpu]["bw"]
    spare = (2 * PARAMS + ATTN_FLOPS_PER_PAIR * ctx) - ridge * KV_PER_TOKEN * ctx
    if spare <= 0:
        return None                                  # the KV reads alone keep intensity below the ridge
    return math.ceil(ridge * PARAMS * FP16 / spare)

for ctx in (100, 500, 1_800):
    limit = (2 * PARAMS + ATTN_FLOPS_PER_PAIR * ctx) / (KV_PER_TOKEN * ctx)   # intensity at huge batch
    cells = []
    for gpu, g in GPUS.items():
        b = batch_for_ridge(ctx, gpu)
        room = int((g["hbm_gb"] * 1e9 - WEIGHTS) // (KV_PER_TOKEN * ctx))
        verdict = "ridge never reached" if b is None else f"batch {b:,}, room {room:,}"
        cells.append(f"{gpu}: {verdict}")
    print(f"context {ctx:>5,}, cap {limit:>5,.0f} FLOP/byte | " + " | ".join(cells))

context   100, cap 1,225 FLOP/byte | L4: batch 600, room 610 | H100 SXM: batch 388, room 4,882
context   500, cap   248 FLOP/byte | L4: ridge never reached | H100 SXM: ridge never reached
context 1,800, cap    72 FLOP/byte | L4: ridge never reached | H100 SXM: ridge never reached


**What you should see:** at 100 tokens the L4 needs batch 600 to reach its ridge, and its memory holds 610, so it only just fits. The H100 needs 388 and has room for 4,882. At 500 tokens or more no batch reaches the ridge, because the KV reads cap the intensity (248 FLOP/byte at 500 tokens, 72 at 1,800). These are ideal counts.

Batching still raises throughput at this context. The next cell decodes the Brightside prompt, 1,800 cached tokens (a 1,600-token prompt from lessons 02 and 03 plus its answer), on the L4 with batch 1, 8 and 32.

In [8]:
print("decode at 1,800 tokens on the L4:")
for b in (1, 8, 32):
    f, by = decode_cost(1800, batch=b)
    bound, ms = phase_limit(f, by, "L4")
    print(f"  batch {b:>2}: {b / (ms / 1000):>5,.0f} tokens/s  ({bound}, step {ms:.1f} ms)")

decode at 1,800 tokens on the L4:
  batch  1:    18 tokens/s  (memory-bound, step 54.1 ms)
  batch  8:   134 tokens/s  (memory-bound, step 59.6 ms)
  batch 32:   408 tokens/s  (memory-bound, step 78.5 ms)


**What you should see:** tokens per second rise with the batch, from about 18 to 134 to 408, and every step stays memory-bound. So for the Brightside prompt decode stays memory-bound at any batch. "Never reaches the ridge" only means never compute-bound. The weights are read once per step, so each extra request adds only its KV bytes. The levers are fewer bytes per step (quantization, shorter context, GQA), not a way to become compute-bound.

### Step 6: a lower precision for the weights, and MBU
FP8 weights take 1 byte per weight, so a decode step reads half the weight bytes. TTFT is compute-bound, so it changes only if the matmuls (the big matrix multiplications that hold most of the FLOPs) also run in FP8. **MBU** (model bandwidth utilisation) is the bytes a step reads, divided by its time, divided by the peak bandwidth. Databricks' example: 14 GB read per token at 14 ms TPOT on a GPU with 2 TB/s peak is 1 TB/s, which is 50% MBU.

In [9]:
f, b_fp16 = decode_cost(1800)
_, b_fp8 = decode_cost(1800, weight_bytes=1)
print(f"L4 decode, batch 1: FP16 weights {phase_limit(f, b_fp16, 'L4')[1]:.1f} ms per token, "
      f"FP8 weights {phase_limit(f, b_fp8, 'L4')[1]:.1f} ms per token")
print(f"MBU check (Databricks example): {14e9 / 0.014 / 2e12:.0%} = 14 GB / 14 ms / 2 TB/s")

L4 decode, batch 1: FP16 weights 54.1 ms per token, FP8 weights 27.5 ms per token
MBU check (Databricks example): 50% = 14 GB / 14 ms / 2 TB/s


**What you should see:** the FP8 weight read is about half the FP16 time. The Databricks check prints 50%.

## At work
- **Databricks** defines MBU as achieved bandwidth over peak bandwidth, and says to measure it on your own deployment rather than trust a spec sheet. → lesson 07 (measure TTFT and TPOT) and lesson 27 (load tests).
- **Cloudflare (May 2026)** reports that separating prefill and decode lowered p90 inter-token latency from about 100 ms with high variance to 20–30 ms (vendor-reported). The two phases stress different limits, so they can run on different pools. → lesson 34 (disaggregated prefill).
- A long prefill holds the GPU's compute while other requests wait for their next decode step. Chunking the prefill spreads that cost. → lesson 12 (continuous batching and chunked prefill).

Sources: Databricks, "LLM Inference Performance Engineering: Best Practices": https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices · Cloudflare, "High-performance LLMs" (May 2026): https://blog.cloudflare.com/high-performance-llms/

## Common mistakes
**1. Timing a step with FLOPs alone.** The roofline gives a floor, not the measured time. Dividing the work by peak maths gives the FLOPs-only estimate from the first cell, which is far too fast for decode. Real engines are slower still; lesson 07 measures by how much.

**2. Treating the ridge as a speed.** The ridge (403 FLOP/byte on the L4, 295 on the H100) is the intensity at which a chip's maths time and memory time are equal. A higher ridge does not make a chip faster. Compare each step's intensity with each chip's ridge.

**3. Passing the whole tuple from `prefill_cost`.** It returns two numbers, FLOPs and bytes. Using the tuple as one number fails.

In [10]:
flops_only = prefill_cost(1600)
print(flops_only / GPUS["L4"]["bw"])   # TypeError: it is a tuple (FLOPs, bytes), not one number

TypeError: unsupported operand type(s) for /: 'tuple' and 'float'

The fix is to unpack it: `flops, bytes_read = prefill_cost(1600)`.

## Try it
Your team considers an H100 for prompts of 4,000 tokens. Write code that prints the prefill's arithmetic intensity, whether it is compute-bound on the H100, and the ideal TTFT in ms. Use `prefill_cost`, `phase_limit` and `GPUS`.

In [11]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [12]:
f, b = prefill_cost(4_000)
bound, ms = phase_limit(f, b, "H100 SXM")
print(f"intensity {f / b:,.0f} FLOP/byte, {bound} on the H100, ideal TTFT {ms:,.0f} ms")

intensity 4,127 FLOP/byte, compute-bound on the H100, ideal TTFT 69 ms


## Recap
- A step takes at least the larger of FLOPs divided by peak FLOP/s, and bytes divided by bandwidth.
- **Prefill** reuses each weight across the prompt, so it is **compute-bound**: TTFT follows the FLOPs.
- **Decode** at batch 1 reads every weight per token, so it is **memory-bound**: TPOT follows bandwidth. FP8 weights cut it.
- Batching shares the weight read, but KV reads cap decode's intensity. At 1,800 tokens no batch reaches the ridge, so decode stays memory-bound while throughput still rises.

**Interview:** "Why is decode slower per token than prefill, with the same weights?" Strong answer: prefill reuses each weight across all prompt tokens, so it is compute-bound; decode reads every weight for one token, about 1 FLOP per byte, so it is memory-bound and TPOT follows bandwidth.